# Sesión 24 · Plotly II: integración con dashboards narrativos
**Visualización I · Universidad Externado de Colombia**  
**Notebook maestro para Google Colab · Docente: Adriana Pineda**

**Pregunta narrativa:** ¿Cómo cambia el volumen de registros de llamadas 123 entre periodos y territorios, y qué patrones merecen una investigación posterior?

**Ruta (6 componentes):** 1. Integración de gráficos; 2. Subplots; 3. Anotaciones y títulos; 4. Filtros y coordinación; 5. Exportación HTML; 6. Integración con Seaborn y preparación del Proyecto 2.

> **Precaución analítica:** una fila es un registro; no equivale necesariamente a un incidente único. Los nombres y columnas reales se verifican antes de cualquier afirmación. Este notebook **no inventa resultados** y no presupone que los archivos de ambos periodos tengan igual cobertura temporal.

**Diseño:** fondo claro, contraste alto, paleta magenta–gris apta para distinción, alineación consistente, proximidad por tema, tipografía legible, uso deliberado del espacio en blanco y ausencia de efectos 3D. No se usa color como único canal para comunicar una categoría.

## 0. Preparar el entorno y cargar las fuentes
En Colab ejecute la celda de carga y seleccione los archivos `llamadas123_2025.xlsx`, `llamadas123.xlsx` y `localidades.csv` (también se admiten nombres con sufijos como `(3)`). Si los archivos ya están en `/content`, no hace falta volver a subirlos.

**Control de calidad:** los archivos y los campos pueden variar. Por eso se diagnostican los nombres de columnas y se detiene la ejecución con un mensaje claro si no existe una variable indispensable.

In [ ]:
# Define la identidad visual del dashboard, estableciendo una paleta de colores coherente con la utilizada en Power BI
# para garantizar uniformidad, legibilidad y jerarquía visual en todos los gráficos.
%pip -q install "plotly>=5.18" "pandas>=2.0" "openpyxl>=3.1" "seaborn>=0.12" "ipywidgets>=8.0"

In [ ]:
import os, re, glob, unicodedata, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# IDENTIDAD VISUAL DEL DASHBOARD DE POWER BI (magenta estimado desde captura)
MAGENTA = '#CB2F60'
MAGENTA_CLARO = '#E8A4BA'
GRIS_OSCURO = '#333333'
GRIS_MEDIO = '#707070'
GRIS_CLARO = '#F2F2F2'
FONDO = '#FFFFFF'
# Alias de trabajo: un mismo color representa un mismo concepto.
AZUL = MAGENTA; AZUL2 = MAGENTA_CLARO; NARANJA = GRIS_MEDIO; GRIS = GRIS_OSCURO
SUAVE = GRIS_CLARO
PALETA = [MAGENTA, GRIS_MEDIO, '#A96A7D', '#969696', '#E8A4BA']
COLOR_FUENTE = {'llamadas123_2025': MAGENTA, 'llamadas123': GRIS_MEDIO}
print('Paleta de referencia Power BI: magenta, gris oscuro, gris medio, gris claro y blanco.')

def normalizar(texto):
    texto = unicodedata.normalize('NFKD', str(texto)).encode('ascii','ignore').decode().lower().strip()
    return re.sub(r'[^a-z0-9]+','_',texto).strip('_')

def tema(fig, titulo, altura=480):
    fig.update_layout(template='plotly_white', title=dict(text=titulo, x=.02, xanchor='left',font=dict(size=19,color=GRIS_OSCURO)),
        font=dict(family='Arial, sans-serif',size=12,color=GRIS), paper_bgcolor=FONDO, plot_bgcolor=FONDO,
        margin=dict(l=70,r=35,t=105,b=65), height=altura, legend=dict(orientation='h',y=1.08,x=0))
    fig.update_xaxes(showgrid=False, linecolor='#D9D9D9',tickfont=dict(size=11))
    fig.update_yaxes(gridcolor='#EFEFEF',zeroline=False,tickfont=dict(size=11))
    return fig

def buscar_archivo(base, extensiones):
    archivos = [p for p in glob.glob('/content/*')+glob.glob('/mnt/data/*')+glob.glob('./*') if os.path.isfile(p)]
    candidatos = []
    for p in archivos:
        nombre = normalizar(os.path.splitext(os.path.basename(p))[0]); ext=os.path.splitext(p)[1].lower()
        if ext in extensiones and (nombre==base or re.fullmatch(re.escape(base)+r'_?[0-9]+',nombre)):
            candidatos.append(p)
    candidatos.sort(key=lambda p:(normalizar(os.path.splitext(os.path.basename(p))[0])!=base,len(p)))
    return candidatos[0] if candidatos else None

rutas = {k:buscar_archivo(k,ext) for k,ext in {
    'llamadas123_2025':('.xlsx','.xls','.csv'), 'llamadas123':('.xlsx','.xls','.csv'), 'localidades':('.csv','.xlsx','.xls')}.items()}
if any(v is None for v in rutas.values()):
    try:
        from google.colab import files
        print('Seleccione los archivos faltantes:',[k for k,v in rutas.items() if v is None]); files.upload()
        rutas = {k:buscar_archivo(k,ext) for k,ext in {
            'llamadas123_2025':('.xlsx','.xls','.csv'), 'llamadas123':('.xlsx','.xls','.csv'), 'localidades':('.csv','.xlsx','.xls')}.items()}
    except ImportError:
        pass
faltantes=[k for k,v in rutas.items() if v is None]
if faltantes: raise FileNotFoundError(f'Faltan archivos: {faltantes}. Súbalos a Colab y vuelva a ejecutar esta celda.')

def leer(p):
    if p.lower().endswith('.csv'):
        for enc in ('utf-8-sig','utf-8','latin-1'):
            try:
                d=pd.read_csv(p,encoding=enc,sep=None,engine='python',skipinitialspace=True)
                if d.shape[1]>1:return d
            except (UnicodeDecodeError,pd.errors.ParserError):continue
        raise ValueError('No se pudo interpretar el CSV: '+p)
    return pd.read_excel(p)

raw25=leer(rutas['llamadas123_2025']); raw=leer(rutas['llamadas123']); raw_loc=leer(rutas['localidades'])
for k,d in [('llamadas123_2025',raw25),('llamadas123',raw),('localidades',raw_loc)]:
    print(f'\n{k}: {d.shape[0]:,} filas, {d.shape[1]} columnas');print(list(d.columns));display(d.head(3))

### Identidad visual compartida con Power BI

El tablero de referencia **Análisis de emergencias – Línea 123** utiliza magenta como acento principal, títulos en gris oscuro, etiquetas en gris medio, líneas tenues y fondo blanco. El valor `#CB2F60` es una **aproximación tomada de la captura**, no una certificación del código original.

**Gestalt y legibilidad:** jerarquía tipográfica, alineación, proximidad entre controles y gráficos, suficiente espacio en blanco, líneas de cuadrícula discretas y ausencia de efectos decorativos. **Uso semántico:** cuando se comparan dos fuentes, magenta identifica `llamadas123_2025` y gris medio identifica `llamadas123`; en vistas de una sola serie se usa magenta. Para prioridades, si se incorporan después, deben definirse colores semánticos constantes.

**Advertencia:** estos datos son registros de llamadas; no se deben rotular automáticamente como emergencias únicas. La identidad gráfica puede compartirse con Power BI sin equiparar los indicadores.

## 1. Diagnóstico y limpieza común (no es todavía la limpieza de cada gráfico)
**Principio:** no borrar registros completos por valores ausentes en variables que no se utilizan. Conservamos copias originales, estandarizamos encabezados y creamos variables analíticas de forma explícita.

Si la detección automática no encuentra la fecha o localidad, revise las columnas impresas arriba y ajuste el diccionario `AJUSTES` en la siguiente celda.

In [ ]:
AJUSTES = {'llamadas123':{'fecha':None,'localidad':None,'categoria':None},
           'llamadas123_2025':{'fecha':None,'localidad':None,'categoria':None},
           'localidades':{'localidad':None}}
# Ejemplo: AJUSTES['llamadas123']['fecha'] = 'fecha_de_creacion'
SINONIMOS={
 'fecha':['fecha','fecha_hora','fecha_llamada','fecha_de_llamada','fecha_registro','fecha_de_registro','fecha_creacion','fecha_de_creacion','fecha_incidente','fecha_inicio','fecha_recepcion','fecha_de_recepcion','created_at','fecha_evento','fecha_de_evento'],
 'localidad':['localidad','nombre_localidad','localidad_nombre','nom_localidad','localidad_incidente','localidad_de_incidente','localidad_evento','nombre_de_localidad','cod_localidad','codigo_localidad'],
 'categoria':['tipo_incidente','tipo_de_incidente','incidente','clasificacion','tipo_evento','tipo_de_evento','tipo_llamada','tipo_de_llamada','prioridad','severidad','categoria','subtipo_incidente']}

def preparar_columnas(df):
    d=df.copy(); d.columns=[normalizar(c) for c in d.columns]
    d=d.loc[:,~d.columns.duplicated()]
    return d

def encontrar(d,clave,origen):
    manual=AJUSTES[origen].get(clave)
    if manual:
        manual=normalizar(manual)
        if manual not in d.columns:raise KeyError(f'{manual} no existe en {origen}. Columnas: {list(d.columns)}')
        return manual
    for s in SINONIMOS[clave]:
        if s in d.columns:return s
    candidatos=[c for c in d.columns if (clave=='fecha' and ('fecha' in c or 'date' in c)) or
                (clave=='localidad' and 'localidad' in c) or
                (clave=='categoria' and any(x in c for x in ('incidente','evento','prioridad','severidad')))]
    return candidatos[0] if candidatos else None

def limpiar_localidad(s):
    x=s.astype('string').str.strip().str.upper().str.replace(r'\s+',' ',regex=True)
    x=x.replace({'':pd.NA,'NAN':pd.NA,'NONE':pd.NA,'SIN INFORMACIÓN':pd.NA,'SIN INFORMACION':pd.NA})
    return x

def crear_base(df,origen):
    d=preparar_columnas(df)
    fecha=encontrar(d,'fecha',origen); loc=encontrar(d,'localidad',origen); cat=encontrar(d,'categoria',origen)
    if fecha is None:raise ValueError(f'No se identificó FECHA en {origen}. Revise AJUSTES. Columnas: {list(d.columns)}')
    # Evitar que números de Excel se interpreten como nanosegundos.
    s=d[fecha]; dt=pd.to_datetime(s,errors='coerce',dayfirst=True)
    numerico=pd.to_numeric(s,errors='coerce'); excel=numerico.between(20000,90000)
    if excel.any():dt.loc[excel]=pd.to_datetime(numerico.loc[excel],unit='D',origin='1899-12-30',errors='coerce')
    d['fecha_analisis']=dt
    d['localidad_analisis']=limpiar_localidad(d[loc]) if loc else pd.Series(pd.NA,index=d.index,dtype='string')
    d['categoria_analisis']=d[cat].astype('string').str.strip() if cat else pd.Series(pd.NA,index=d.index,dtype='string')
    d['fuente']=origen
    print(origen,'| fecha:',fecha,'| localidad:',loc,'| categoría:',cat,'| fechas válidas:',d.fecha_analisis.notna().sum())
    return d

base25=crear_base(raw25,'llamadas123_2025'); base=crear_base(raw,'llamadas123')
loc_ref=preparar_columnas(raw_loc); campo_loc=encontrar(loc_ref,'localidad','localidades')
if campo_loc:
    loc_ref['localidad_analisis']=limpiar_localidad(loc_ref[campo_loc]); print('Localidades de referencia:',loc_ref.localidad_analisis.dropna().nunique())
else:
    print('La tabla localidades no contiene un nombre identificable; se conserva para inspección, sin uniones especulativas.')

# Solo apilamos las fuentes: NO hacemos merge por localidad porque generaría duplicación de filas
conjunto=pd.concat([base25,base],ignore_index=True,sort=False)
print('Registros combinados:',len(conjunto));print('Rangos de fechas por fuente:')
display(conjunto.groupby('fuente').fecha_analisis.agg(['min','max','count']))

## 2. Componente 1 · Integración de visualizaciones
**Gráfico 1: evolución mensual por fuente.** Antes de graficar: filtrar fechas válidas, construir el periodo mensual y agregar conteos. No rellenar con ceros los meses que no pertenecen a la cobertura de una fuente.

In [ ]:
# LIMPIEZA ESPECÍFICA DEL GRÁFICO 1

# Seleccionar únicamente los registros con fechas válidas
mensual = conjunto.loc[
    conjunto['fecha_analisis'].notna(),
    ['fecha_analisis']
].copy()

# Convertir las fechas al primer día de cada mes
mensual['mes'] = (
    mensual['fecha_analisis']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Contar los registros de cada mes, unificando ambas fuentes
mensual = (
    mensual.groupby('mes')
    .size()
    .reset_index(name='registros')
    .sort_values('mes')
)

# Verificar que existan datos para graficar
assert not mensual.empty, "No hay fechas válidas para el gráfico mensual."

# GRÁFICO 1: EVOLUCIÓN MENSUAL DE REGISTROS

fig1 = px.line(
    mensual,
    x='mes',
    y='registros',
    markers=True,
    labels={
        'mes': 'Mes',
        'registros': 'Total de registros'
    },
    hover_data={
        'mes': '%B %Y',
        'registros': ':,'
    }
)

# Aplicar la identidad visual utilizada en Power BI
tema(fig1, '01 · Evolución mensual de registros')

fig1.update_traces(
    line_color='#CB2F60',
    line_width=3,
    marker_size=8
)

# Mostrar los meses en orden cronológico
fig1.update_xaxes(
    dtick='M1',
    tickformat='%b %Y'
)

# Anotar julio de 2026 cuando existan registros
julio = mensual[
    mensual['mes'] == pd.Timestamp('2026-07-01')
]

if not julio.empty:
    valor_julio = int(julio['registros'].iloc[0])

    fig1.add_annotation(
        x=pd.Timestamp('2026-07-01'),
        y=valor_julio,
        text=f'Julio: {valor_julio} registros<br>Verificar cobertura temporal',
        showarrow=True,
        arrowhead=2,
        ax=-100,
        ay=-75,
        font=dict(
            size=12,
            color='#333333'
        ),
        bgcolor='white',
        bordercolor='#F2F2F2',
        borderpad=6
    )

# Configurar la interacción y los márgenes
fig1.update_layout(
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=60, r=50, t=90, b=70)
)

fig1.show()

**Lectura crítica:** los meses observados pueden ser distintos entre archivos. No interpretar diferencias de totales como crecimiento sin verificar que las ventanas temporales sean comparables.

**Gráfico 2: distribución territorial.** Limpieza previa: eliminar únicamente localidades faltantes, estandarizar nombres y agrupar categorías pequeñas para facilitar la lectura.

In [ ]:
# ============================================================
# GRÁFICO 2. LOCALIDADES CON MÁS REGISTROS
# Limpieza, consolidación y visualización
# ============================================================

import pandas as pd
import plotly.express as px
import unicodedata
import re

# 1. Seleccionar los datos originales
territorial = conjunto[['localidad_analisis']].copy()

# 2. Normalizar nombres de localidades
def normalizar_localidad(valor):
    if pd.isna(valor):
        return None

    valor = str(valor).strip().upper()
    valor = unicodedata.normalize('NFKD', valor)

    valor = ''.join(
        c for c in valor
        if not unicodedata.combining(c)
    )

    valor = re.sub(r'[^A-Z ]', '', valor)
    valor = re.sub(r'\s+', ' ', valor).strip()

    return valor if valor else None


territorial['localidad_normalizada'] = (
    territorial['localidad_analisis']
    .apply(normalizar_localidad)
)

# 3. Catálogo de localidades válidas de Bogotá
localidades_validas = [
    'USAQUEN', 'CHAPINERO', 'SANTA FE', 'SAN CRISTOBAL',
    'USME', 'TUNJUELITO', 'BOSA', 'KENNEDY',
    'FONTIBON', 'ENGATIVA', 'SUBA', 'BARRIOS UNIDOS',
    'TEUSAQUILLO', 'LOS MARTIRES', 'ANTONIO NARINO',
    'PUENTE ARANDA', 'LA CANDELARIA',
    'RAFAEL URIBE URIBE', 'CIUDAD BOLIVAR', 'SUMAPAZ'
]

# 4. Corregir variantes identificadas en el diagnóstico
# Las claves corresponden a los nombres ya normalizados.
correcciones = {
    'ENGATIV': 'ENGATIVA',
    'ENGATIVC': 'ENGATIVA',
    'ENGATIVM': 'ENGATIVA',

    'CIUDAD BOLOVAR': 'CIUDAD BOLIVAR',
    'CIUDAD BOLEVAR': 'CIUDAD BOLIVAR',
    'CIUDAD BOLVAR': 'CIUDAD BOLIVAR',

    'USAQUN': 'USAQUEN',
    'USAQUUN': 'USAQUEN',

    'ANTONIO NARIO': 'ANTONIO NARINO',
    'ANTONIO NARINOY': 'ANTONIO NARINO',

    'SAN CRISTIBAL': 'SAN CRISTOBAL',
    'SAN CRISTABAL': 'SAN CRISTOBAL',
    'SAN CRISTBAL': 'SAN CRISTOBAL',

    'FONTIBIN': 'FONTIBON',
    'FONTIBTN': 'FONTIBON',
    'FONTIBAN': 'FONTIBON',

    'LOS MCRT IRES': 'LOS MARTIRES',
    'LOS MCRTIRES': 'LOS MARTIRES',
    'LOS MCTIRES': 'LOS MARTIRES',
    'LOS MRTIRES': 'LOS MARTIRES',

    'BARRIOS UNIDOSY': 'BARRIOS UNIDOS'
}

# 5. Aplicar las correcciones
territorial['localidad_limpia'] = (
    territorial['localidad_normalizada']
    .replace(correcciones)
)

# 6. Identificar categorías que aún requieren revisión
pendientes = (
    territorial.loc[
        territorial['localidad_limpia'].notna() &
        ~territorial['localidad_limpia'].isin(localidades_validas),
        'localidad_limpia'
    ]
    .value_counts()
)

print("LOCALIDADES PENDIENTES DE REVISIÓN:")
print(
    pendientes.to_string()
    if not pendientes.empty else "Ninguna"
)

# 7. Conservar únicamente localidades válidas
territorial = territorial[
    territorial['localidad_limpia'].isin(localidades_validas)
].copy()

# 8. Calcular frecuencias por localidad
conteos = (
    territorial['localidad_limpia']
    .value_counts()
    .sort_values(ascending=False)
)

print("\nLOCALIDADES CORREGIDAS:")
print(conteos.to_string())

# 9. Verificar la consolidación de Engativá
total_engativa = conteos.get('ENGATIVA', 0)

print(
    f"\nTotal consolidado de Engativá: "
    f"{total_engativa:,}"
)

# 10. Seleccionar las diez localidades con más registros
if conteos.empty:
    print("No hay localidades válidas para construir el gráfico.")

else:
    top = conteos.head(10).sort_values()

    # 11. Construir el gráfico de barras horizontales
    fig2 = px.bar(
        x=top.values,
        y=top.index,
        orientation='h',
        text=top.values,
        labels={
            'x': 'Total de registros',
            'y': 'Localidad'
        }
    )

    # 12. Aplicar la identidad visual de Power BI
    tema(
        fig2,
        '02 · Localidades con más registros',
        altura=520
    )

    fig2.update_traces(
        marker_color='#CB2F60',
        texttemplate='%{text:,.0f}',
        textposition='outside',
        cliponaxis=False,
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Total de registros: %{x:,.0f}'
            '<extra></extra>'
        )
    )

    # 13. Mejorar la legibilidad del gráfico
    fig2.update_layout(
        showlegend=False,
        margin=dict(l=175, r=100, t=105, b=65),
        xaxis=dict(
            title='Total de registros',
            rangemode='tozero',
            showgrid=True,
            gridcolor='#F2F2F2'
        ),
        yaxis=dict(
            title=None,
            showgrid=False
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    # 14. Mostrar el gráfico
    fig2.show()

## 3. Componente 2 · Subplots
**Gráfico 3: dos vistas, una lectura.** Limpieza previa independiente para cada panel: agregar por mes y calcular la distribución por día de semana solo con fechas válidas. Evitamos escalas compartidas entre magnitudes diferentes.

In [ ]:
# ============================================================
# GRÁFICO 3. PATRONES TEMPORALES DE LOS REGISTROS
# Subplots: evolución mensual y promedio por día de semana
# ============================================================

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. LIMPIEZA ESPECÍFICA DE LOS DATOS
# ------------------------------------------------------------

# Seleccionar únicamente la variable de fecha
temporal = conjunto[['fecha_analisis']].copy()

# Convertir los valores a fechas y controlar posibles errores
temporal['fecha_analisis'] = pd.to_datetime(
    temporal['fecha_analisis'],
    errors='coerce'
)

# Eliminar registros sin fechas válidas
temporal = temporal.dropna(subset=['fecha_analisis']).copy()

# Verificar que existan datos disponibles
assert not temporal.empty, "No hay fechas válidas para el análisis."

# Obtener la fecha sin componente de hora
temporal['fecha'] = temporal['fecha_analisis'].dt.normalize()


# ------------------------------------------------------------
# 2. PREPARAR LA EVOLUCIÓN MENSUAL
# ------------------------------------------------------------

# Identificar el mes correspondiente a cada registro
temporal['mes'] = (
    temporal['fecha_analisis']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Calcular el total de registros por mes
mensual = (
    temporal.groupby('mes')
    .size()
    .reset_index(name='registros')
    .sort_values('mes')
)

# ------------------------------------------------------------
# 3. PREPARAR EL PROMEDIO POR DÍA DE SEMANA
# ------------------------------------------------------------

# Identificar el día de semana de cada registro
# Python: lunes = 0, domingo = 6
temporal['dia_semana'] = temporal['fecha'].dt.dayofweek

# Contar los registros por día de semana
registros_semana = (
    temporal.groupby('dia_semana')
    .size()
    .reindex(range(7), fill_value=0)
)

# Construir el calendario del período analizado
fecha_inicio = temporal['fecha'].min()
fecha_fin = temporal['fecha'].max()

calendario = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq='D'
)

# Contar cuántas veces aparece cada día de semana
dias_calendario = (
    pd.Series(calendario.dayofweek)
    .value_counts()
    .reindex(range(7), fill_value=0)
)

# Calcular el promedio de registros por día de semana
promedio_semana = (
    registros_semana
    .div(dias_calendario.replace(0, float('nan')))
)

# Crear una tabla organizada para el gráfico
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

semanal = pd.DataFrame({
    'dia': dias,
    'promedio': promedio_semana.values,
    'registros': registros_semana.values,
    'numero_dias': dias_calendario.values
})

# Eliminar días sin observaciones en el calendario
semanal = semanal.dropna(subset=['promedio'])

# Mostrar los resultados para verificar los cálculos
print("PROMEDIO DE REGISTROS POR DÍA DE SEMANA:")
print(semanal.to_string(index=False))


# ------------------------------------------------------------
# 4. CONSTRUIR LOS DOS SUBPLOTS
# ------------------------------------------------------------

fig3 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        'Evolución mensual de registros',
        'Promedio por día de semana'
    ),
    horizontal_spacing=0.12
)


# ------------------------------------------------------------
# 5. GRÁFICO IZQUIERDO: EVOLUCIÓN MENSUAL
# ------------------------------------------------------------

fig3.add_trace(
    go.Scatter(
        x=mensual['mes'],
        y=mensual['registros'],
        mode='lines+markers',
        name='Registros mensuales',
        line=dict(
            color='#CB2F60',
            width=3
        ),
        marker=dict(size=8),
        hovertemplate=(
            'Mes: %{x|%B %Y}<br>'
            'Registros: %{y:,.0f}'
            '<extra></extra>'
        )
    ),
    row=1,
    col=1
)


# ------------------------------------------------------------
# 6. GRÁFICO DERECHO: PROMEDIO POR DÍA DE SEMANA
# ------------------------------------------------------------

fig3.add_trace(
    go.Bar(
        x=semanal['dia'],
        y=semanal['promedio'],
        name='Promedio diario',
        marker_color='#707070',
        customdata=semanal[['registros', 'numero_dias']].to_numpy(),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Promedio: %{y:,.1f} registros<br>'
            'Total de registros: %{customdata[0]:,.0f}<br>'
            'Número de días: %{customdata[1]:,.0f}'
            '<extra></extra>'
        )
    ),
    row=1,
    col=2
)


# ------------------------------------------------------------
# 7. AGREGAR ANOTACIÓN SOBRE JULIO DE 2026
# ------------------------------------------------------------

julio = mensual[
    mensual['mes'] == pd.Timestamp('2026-07-01')
]

if not julio.empty:

    valor_julio = int(julio['registros'].iloc[0])

    fig3.add_annotation(
        x=pd.Timestamp('2026-07-01'),
        y=valor_julio,
        text=(
            f'Julio: {valor_julio} registros'
            '<br>Verificar cobertura temporal'
        ),
        showarrow=True,
        arrowhead=2,
        ax=-95,
        ay=-75,
        font=dict(
            size=11,
            color='#333333'
        ),
        bgcolor='white',
        bordercolor='#F2F2F2',
        borderpad=5,
        row=1,
        col=1
    )


# ------------------------------------------------------------
# 8. APLICAR LA IDENTIDAD VISUAL DEL DASHBOARD
# ------------------------------------------------------------

tema(
    fig3,
    '03 · Patrones temporales de los registros',
    altura=570
)

# Configurar los ejes del gráfico mensual
fig3.update_xaxes(
    title_text='Mes',
    dtick='M1',
    tickformat='%b %Y',
    row=1,
    col=1
)

fig3.update_yaxes(
    title_text='Total de registros',
    rangemode='tozero',
    row=1,
    col=1
)

# Configurar los ejes del gráfico semanal
fig3.update_xaxes(
    title_text='Día de semana',
    categoryorder='array',
    categoryarray=dias,
    row=1,
    col=2
)

fig3.update_yaxes(
    title_text='Promedio de registros por día',
    rangemode='tozero',
    row=1,
    col=2
)

# Ajustar márgenes, separación y fondo
fig3.update_layout(
    showlegend=False,
    margin=dict(
        l=75,
        r=55,
        t=125,
        b=75
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='closest'
)

# Mostrar el gráfico interactivo
fig3.show()

## 4. Componente 3 · Anotaciones, títulos y narrativa
**Gráfico 4: destacar un hallazgo sin exagerarlo.** Limpieza previa: agrupar por mes, verificar fechas y detectar el máximo observado. La anotación expresa un hecho descriptivo, no una explicación causal.

In [ ]:
# ============================================================
# GRÁFICO 4. IDENTIFICACIÓN DEL MÁXIMO MENSUAL OBSERVADO
# Gráfico de área con anotaciones y narrativa visual
# ============================================================

import pandas as pd
import plotly.graph_objects as go

# ------------------------------------------------------------
# 1. LIMPIEZA ESPECÍFICA DE LOS DATOS
# ------------------------------------------------------------

# Seleccionar únicamente la variable necesaria para el gráfico
datos_g4 = conjunto[['fecha_analisis']].copy()

# Convertir los valores a formato de fecha
# Los valores que no puedan convertirse se reemplazan por NaT
datos_g4['fecha_analisis'] = pd.to_datetime(
    datos_g4['fecha_analisis'],
    errors='coerce'
)

# Eliminar registros con fechas faltantes o inválidas
datos_g4 = datos_g4.dropna(
    subset=['fecha_analisis']
).copy()

# Verificar que existan datos válidos
assert not datos_g4.empty, (
    "No hay fechas válidas para construir el gráfico 4."
)

# ------------------------------------------------------------
# 2. CALCULAR EL TOTAL DE REGISTROS POR MES
# ------------------------------------------------------------

# Convertir las fechas al primer día de cada mes
datos_g4['mes'] = (
    datos_g4['fecha_analisis']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Agrupar los registros por mes
mensual_g4 = (
    datos_g4.groupby('mes')
    .size()
    .reset_index(name='registros')
    .sort_values('mes')
)

# ------------------------------------------------------------
# 3. IDENTIFICAR AUTOMÁTICAMENTE EL MÁXIMO MENSUAL
# ------------------------------------------------------------

# Localizar la fila con el mayor número de registros
fila_maximo = mensual_g4.loc[
    mensual_g4['registros'].idxmax()
]

# Extraer el mes y el valor máximo
mes_maximo = fila_maximo['mes']
valor_maximo = int(fila_maximo['registros'])

# Crear nombres de meses en español
meses_es = [
    'enero', 'febrero', 'marzo', 'abril',
    'mayo', 'junio', 'julio', 'agosto',
    'septiembre', 'octubre', 'noviembre', 'diciembre'
]

nombre_mes = meses_es[mes_maximo.month - 1]
anio_maximo = mes_maximo.year

# Crear el título narrativo automáticamente
titulo_g4 = (
    f'04 · {nombre_mes.capitalize()} de {anio_maximo} '
    f'registró el mayor volumen mensual: {valor_maximo:,} registros'
)

print("MÁXIMO MENSUAL IDENTIFICADO:")
print(f"Mes: {nombre_mes.capitalize()} de {anio_maximo}")
print(f"Registros: {valor_maximo:,}")

# ------------------------------------------------------------
# 4. CONSTRUIR EL GRÁFICO DE ÁREA
# ------------------------------------------------------------

fig4 = go.Figure()

fig4.add_trace(
    go.Scatter(
        x=mensual_g4['mes'],
        y=mensual_g4['registros'],
        mode='lines+markers',
        fill='tozeroy',

        # Identidad visual de Power BI
        line=dict(
            color='#CB2F60',
            width=3
        ),

        marker=dict(
            size=7,
            color='#CB2F60'
        ),

        # Relleno magenta con transparencia
        fillcolor='rgba(203, 47, 96, 0.12)',

        name='Registros mensuales',

        hovertemplate=(
            'Mes: %{x|%B %Y}<br>'
            'Registros: %{y:,.0f}'
            '<extra></extra>'
        )
    )
)

# ------------------------------------------------------------
# 5. APLICAR LA IDENTIDAD VISUAL DEL DASHBOARD
# ------------------------------------------------------------

tema(
    fig4,
    titulo_g4,
    altura=570
)

# ------------------------------------------------------------
# 6. DESTACAR EL MÁXIMO MENSUAL
# ------------------------------------------------------------

# Incorporar un marcador adicional sobre el valor máximo
fig4.add_trace(
    go.Scatter(
        x=[mes_maximo],
        y=[valor_maximo],
        mode='markers',

        marker=dict(
            color='#CB2F60',
            size=13,
            line=dict(
                color='white',
                width=2
            )
        ),

        showlegend=False,

        hovertemplate=(
            '<b>Máximo mensual</b><br>'
            f'{nombre_mes.capitalize()} de {anio_maximo}<br>'
            f'{valor_maximo:,} registros'
            '<extra></extra>'
        )
    )
)

# Agregar una anotación que destaque el hallazgo
fig4.add_annotation(
    x=mes_maximo,
    y=valor_maximo,

    text=f'<b>Máximo observado: {valor_maximo:,}</b>',

    showarrow=True,
    arrowhead=2,
    arrowwidth=1.5,
    arrowcolor='#CB2F60',

    # Desplazar la anotación hacia arriba
    ax=0,
    ay=-65,

    font=dict(
        size=12,
        color='#333333'
    ),

    bgcolor='white',
    bordercolor='#F2F2F2',
    borderwidth=1,
    borderpad=7
)

# ------------------------------------------------------------
# 7. ADVERTENCIA SOBRE LA COBERTURA DE JULIO DE 2026
# ------------------------------------------------------------

julio_g4 = mensual_g4[
    mensual_g4['mes'] == pd.Timestamp('2026-07-01')
]

if not julio_g4.empty:

    registros_julio = int(
        julio_g4['registros'].iloc[0]
    )

    # Incorporar una nota informativa debajo del gráfico
    fig4.add_annotation(
        x=0,
        y=-0.23,
        xref='paper',
        yref='paper',

        text=(
            f'<b>Nota:</b> Julio de 2026 presenta '
            f'{registros_julio:,} registros. '
            'Se recomienda verificar la cobertura temporal '
            'antes de interpretar la disminución.'
        ),

        showarrow=False,

        xanchor='left',
        align='left',

        font=dict(
            size=11,
            color='#707070'
        )
    )

# ------------------------------------------------------------
# 8. CONFIGURAR LOS EJES
# ------------------------------------------------------------

# Eje horizontal: meses
fig4.update_xaxes(
    title_text='Mes',
    dtick='M1',
    tickformat='%b %Y',
    showgrid=False,
    showline=True,
    linecolor='#D9D9D9'
)

# Eje vertical: número de registros
fig4.update_yaxes(
    title_text='Total de registros',
    rangemode='tozero',

    # Reservar espacio superior para la anotación
    range=[0, valor_maximo * 1.30],

    showgrid=True,
    gridcolor='#F2F2F2',
    zeroline=False
)

# ------------------------------------------------------------
# 9. AJUSTAR LA PRESENTACIÓN GENERAL
# ------------------------------------------------------------

fig4.update_layout(
    showlegend=False,

    margin=dict(
        l=75,
        r=55,
        t=120,
        b=130
    ),

    plot_bgcolor='white',
    paper_bgcolor='white',

    hovermode='x unified'
)

# ------------------------------------------------------------
# 10. MOSTRAR EL GRÁFICO INTERACTIVO
# ------------------------------------------------------------

fig4.show()

## 5. Componente 4 · Mini-dashboard con filtros coordinados
**Objetivo:** cambiar la fuente con un menú y actualizar **simultáneamente** el indicador, la serie temporal y el ranking de localidades. La coordinación se implementa con un `updatemenus` nativo de Plotly, por lo que **también funciona en el HTML exportado**, sin servidor ni Dash.

**Limitación explicada:** los gráficos Plotly independientes no se filtran mutuamente automáticamente al hacer clic. Aquí un único control actualiza las tres vistas; para filtros cruzados arbitrarios con clics entre gráficos haría falta Dash, Streamlit u otra capa de aplicación.

**Limpieza previa del dashboard:** fechas y localidades se preparan por separado, se normalizan etiquetas y se calculan agregados por fuente. Los periodos no observados no se imputan como cero.

In [ ]:
# ============================================================
# GRÁFICO 5. DASHBOARD NARRATIVO INTERACTIVO
# Análisis de registros – Línea 123
#
# Componentes:
# 1. Limpieza de fechas, fuentes y localidades
# 2. Tres indicadores KPI
# 3. Evolución mensual
# 4. Ranking de localidades
# 5. Filtros coordinados
# 6. Diseño visual y advertencias temporales
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata

import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# 1. IDENTIDAD VISUAL DEL DASHBOARD
# ============================================================

# Colores utilizados en el dashboard de Power BI
MAGENTA = '#CB2F60'
GRIS_OSCURO = '#333333'
GRIS_MEDIO = '#707070'
GRIS_CLARO = '#F2F2F2'
BLANCO = '#FFFFFF'

# Nombres de los meses en español
MESES_ES = [
    'enero', 'febrero', 'marzo', 'abril',
    'mayo', 'junio', 'julio', 'agosto',
    'septiembre', 'octubre', 'noviembre',
    'diciembre'
]


# ============================================================
# 2. LIMPIEZA ESPECÍFICA DEL DASHBOARD
# ============================================================

# Seleccionar únicamente las variables necesarias
# sin modificar la base de datos original.

datos_dash = conjunto[
    ['fuente', 'fecha_analisis', 'localidad_analisis']
].copy()

# Estandarizar los nombres de las fuentes
datos_dash['fuente'] = (
    datos_dash['fuente']
    .astype('string')
    .str.strip()
)

# Convertir las fechas a formato datetime
datos_dash['fecha_analisis'] = pd.to_datetime(
    datos_dash['fecha_analisis'],
    errors='coerce'
)

# Crear una variable correspondiente al mes
datos_dash['mes'] = (
    datos_dash['fecha_analisis']
    .dt.to_period('M')
    .dt.to_timestamp()
)


# ============================================================
# 3. NORMALIZACIÓN DE LOCALIDADES
# ============================================================

# Esta función corrige diferencias de mayúsculas,
# tildes, espacios y caracteres especiales.

def normalizar_localidad_dash(valor):

    if pd.isna(valor):
        return None

    valor = str(valor).strip().upper()

    # Descomponer caracteres acentuados
    valor = unicodedata.normalize('NFKD', valor)

    # Eliminar tildes y marcas diacríticas
    valor = ''.join(
        caracter for caracter in valor
        if not unicodedata.combining(caracter)
    )

    # Conservar letras y espacios
    valor = re.sub(r'[^A-Z ]', '', valor)

    # Eliminar espacios repetidos
    valor = re.sub(r'\s+', ' ', valor).strip()

    return valor if valor else None


# Aplicar la función a los registros originales
datos_dash['localidad_normalizada'] = (
    datos_dash['localidad_analisis']
    .apply(normalizar_localidad_dash)
)


# ============================================================
# 4. CORRECCIÓN DE VARIANTES DE LOCALIDADES
# ============================================================

# Se utilizan las variantes identificadas y corregidas
# durante la construcción del gráfico 2.

correcciones_dash = {

    # Engativá
    'ENGATIV': 'ENGATIVA',
    'ENGATIVC': 'ENGATIVA',
    'ENGATIVM': 'ENGATIVA',

    # Ciudad Bolívar
    'CIUDAD BOLOVAR': 'CIUDAD BOLIVAR',
    'CIUDAD BOLEVAR': 'CIUDAD BOLIVAR',
    'CIUDAD BOLVAR': 'CIUDAD BOLIVAR',

    # Usaquén
    'USAQUN': 'USAQUEN',
    'USAQUUN': 'USAQUEN',

    # Antonio Nariño
    'ANTONIO NARIO': 'ANTONIO NARINO',
    'ANTONIO NARINOY': 'ANTONIO NARINO',

    # San Cristóbal
    'SAN CRISTIBAL': 'SAN CRISTOBAL',
    'SAN CRISTABAL': 'SAN CRISTOBAL',
    'SAN CRISTBAL': 'SAN CRISTOBAL',

    # Fontibón
    'FONTIBIN': 'FONTIBON',
    'FONTIBTN': 'FONTIBON',
    'FONTIBAN': 'FONTIBON',

    # Los Mártires
    'LOS MCRTIRES': 'LOS MARTIRES',
    'LOS MCTIRES': 'LOS MARTIRES',
    'LOS MRTIRES': 'LOS MARTIRES',

    # Barrios Unidos
    'BARRIOS UNIDOSY': 'BARRIOS UNIDOS'
}

# Aplicar las correcciones
datos_dash['localidad_limpia'] = (
    datos_dash['localidad_normalizada']
    .replace(correcciones_dash)
)


# ============================================================
# 5. VALIDACIÓN DE LOCALIDADES
# ============================================================

# Catálogo de las 20 localidades de Bogotá
localidades_validas = [
    'USAQUEN',
    'CHAPINERO',
    'SANTA FE',
    'SAN CRISTOBAL',
    'USME',
    'TUNJUELITO',
    'BOSA',
    'KENNEDY',
    'FONTIBON',
    'ENGATIVA',
    'SUBA',
    'BARRIOS UNIDOS',
    'TEUSAQUILLO',
    'LOS MARTIRES',
    'ANTONIO NARINO',
    'PUENTE ARANDA',
    'LA CANDELARIA',
    'RAFAEL URIBE URIBE',
    'CIUDAD BOLIVAR',
    'SUMAPAZ'
]

# Identificar los nombres no reconocidos
pendientes = (
    datos_dash.loc[
        datos_dash['localidad_limpia'].notna()
        & ~datos_dash['localidad_limpia'].isin(
            localidades_validas
        ),
        'localidad_limpia'
    ]
    .value_counts()
)

print("LOCALIDADES PENDIENTES DE REVISIÓN:")

if pendientes.empty:
    print("Ninguna")
else:
    print(pendientes.to_string())

# Los nombres no reconocidos no se incluyen en
# los rankings territoriales, pero sus registros
# permanecen en el total general.

datos_dash.loc[
    ~datos_dash['localidad_limpia'].isin(
        localidades_validas
    ),
    'localidad_limpia'
] = None

# Verificar la consolidación de Engativá
total_engativa = (
    datos_dash['localidad_limpia']
    .eq('ENGATIVA')
    .sum()
)

print(
    f"\nRegistros consolidados de Engativá: "
    f"{total_engativa:,}"
)


# ============================================================
# 6. DEFINIR LOS FILTROS
# ============================================================

# Identificar las fuentes disponibles
fuentes_disponibles = sorted(
    datos_dash['fuente']
    .dropna()
    .unique()
    .tolist()
)

fuentes = (
    ['Todas las fuentes']
    + fuentes_disponibles
)

# Identificar las localidades válidas presentes
localidades_disponibles = sorted(
    datos_dash['localidad_limpia']
    .dropna()
    .unique()
    .tolist()
)

localidades_filtro = (
    ['Todas las localidades']
    + localidades_disponibles
)

# Construir las combinaciones de filtros
combinaciones = [
    (fuente, localidad)
    for fuente in fuentes
    for localidad in localidades_filtro
]


# ============================================================
# 7. FUNCIÓN PARA CALCULAR LOS RESULTADOS
# ============================================================

def calcular_resumen_dash(fuente, localidad):

    # Aplicar filtro por fuente
    if fuente == 'Todas las fuentes':
        tmp = datos_dash.copy()
    else:
        tmp = datos_dash.loc[
            datos_dash['fuente'].eq(fuente)
        ].copy()

    # Aplicar filtro por localidad
    if localidad != 'Todas las localidades':
        tmp = tmp.loc[
            tmp['localidad_limpia'].eq(localidad)
        ].copy()

    # KPI 1: total de registros
    total = len(tmp)

    # Calcular registros mensuales
    mensual = (
        tmp.dropna(subset=['mes'])
        .groupby('mes')
        .size()
        .reset_index(name='registros')
        .sort_values('mes')
    )

    # Calcular registros por localidad
    conteos = (
        tmp['localidad_limpia']
        .dropna()
        .value_counts()
    )

    # KPI 2: localidad principal
    localidad_principal = (
        conteos.index[0]
        if not conteos.empty
        else 'Sin información'
    )

    # KPI 3: mes de mayor volumen
    if not mensual.empty:

        fila_max = mensual.loc[
            mensual['registros'].idxmax()
        ]

        fecha_max = fila_max['mes']

        mes_principal = (
            f"{MESES_ES[fecha_max.month - 1].capitalize()} "
            f"de {fecha_max.year}"
        )

    else:
        mes_principal = 'Sin información'

    # Seleccionar las ocho localidades principales
    ranking = (
        conteos.head(8)
        .sort_values()
    )

    # Comprobar si hay registros en julio de 2026
    julio_2026 = mensual.loc[
        mensual['mes'].eq(
            pd.Timestamp('2026-07-01')
        ),
        'registros'
    ]

    registros_julio = (
        int(julio_2026.iloc[0])
        if not julio_2026.empty
        else None
    )

    # Crear una advertencia específica para el filtro
    if registros_julio is not None:

        nota = (
            f"<b>Nota:</b> Julio de 2026 presenta "
            f"{registros_julio:,} registros en la "
            f"selección actual. Verificar la cobertura "
            f"temporal antes de interpretar la disminución."
        )

    else:

        nota = (
            "<b>Nota:</b> Los resultados corresponden "
            "a los registros disponibles para los "
            "filtros seleccionados."
        )

    return {
        'total': total,
        'localidad_principal': localidad_principal,
        'mes_principal': mes_principal,
        'mensual': mensual,
        'ranking': ranking,
        'nota': nota
    }


# ============================================================
# 8. CALCULAR LOS ESTADOS DEL DASHBOARD
# ============================================================

# Cada combinación contiene los indicadores
# y los datos necesarios para ambos gráficos.

resumen_dash = {
    (fuente, localidad): calcular_resumen_dash(
        fuente,
        localidad
    )
    for fuente, localidad in combinaciones
}

# Definir los filtros iniciales
estado_inicial = (
    'Todas las fuentes',
    'Todas las localidades'
)


# ============================================================
# 9. CONSTRUIR LA ESTRUCTURA DEL DASHBOARD
# ============================================================

# Se utilizan dos filas:
#
# Fila 1: tres indicadores KPI
# Fila 2: evolución mensual y ranking territorial
#
# Se reduce la separación vertical para conseguir
# una distribución más compacta.

dashboard = make_subplots(

    rows=2,
    cols=3,

    specs=[
        [
            {'type': 'indicator'},
            {'type': 'indicator'},
            {'type': 'indicator'}
        ],
        [
            {'type': 'xy', 'colspan': 2},
            None,
            {'type': 'xy'}
        ]
    ],

    subplot_titles=(
        'Total de registros',
        'Localidad principal',
        'Mes de mayor volumen',
        'Evolución mensual',
        'Localidades principales'
    ),

    row_heights=[0.20, 0.80],

    vertical_spacing=0.12,
    horizontal_spacing=0.10
)


# ============================================================
# 10. CREAR LOS KPI Y GRÁFICOS DE CADA ESTADO
# ============================================================

# Cada combinación genera cinco trazas:
#
# 1. Total de registros
# 2. Localidad principal
# 3. Mes de mayor volumen
# 4. Evolución mensual
# 5. Ranking de localidades

indices_estados = {}

for fuente, localidad in combinaciones:

    r = resumen_dash[(fuente, localidad)]

    visible = (
        (fuente, localidad) == estado_inicial
    )

    inicio = len(dashboard.data)

    # --------------------------------------------------------
    # KPI 1. TOTAL DE REGISTROS
    # --------------------------------------------------------

    dashboard.add_trace(
        go.Indicator(
            mode='number',
            value=r['total'],

            number=dict(
                valueformat=',',
                font=dict(
                    size=37,
                    color=GRIS_OSCURO
                )
            ),

            visible=visible
        ),
        row=1,
        col=1
    )

    # --------------------------------------------------------
    # KPI 2. LOCALIDAD PRINCIPAL
    # --------------------------------------------------------

    # Se utiliza el título del indicador para
    # representar el nombre de la localidad.

    dashboard.add_trace(
        go.Indicator(
            mode='number',
            value=0,

            number=dict(
                font=dict(
                    size=1,
                    color=BLANCO
                )
            ),

            title=dict(
                text=r['localidad_principal'],
                font=dict(
                    size=19,
                    color=GRIS_OSCURO
                )
            ),

            visible=visible
        ),
        row=1,
        col=2
    )

    # --------------------------------------------------------
    # KPI 3. MES DE MAYOR VOLUMEN
    # --------------------------------------------------------

    dashboard.add_trace(
        go.Indicator(
            mode='number',
            value=0,

            number=dict(
                font=dict(
                    size=1,
                    color=BLANCO
                )
            ),

            title=dict(
                text=r['mes_principal'],
                font=dict(
                    size=19,
                    color=GRIS_OSCURO
                )
            ),

            visible=visible
        ),
        row=1,
        col=3
    )

    # --------------------------------------------------------
    # GRÁFICO A. EVOLUCIÓN MENSUAL
    # --------------------------------------------------------

    dashboard.add_trace(
        go.Scatter(

            x=r['mensual']['mes'],
            y=r['mensual']['registros'],

            mode='lines+markers',

            line=dict(
                color=MAGENTA,
                width=3
            ),

            marker=dict(
                color=MAGENTA,
                size=7
            ),

            visible=visible,

            hovertemplate=(
                'Mes: %{x|%Y-%m}<br>'
                'Registros: %{y:,.0f}'
                '<extra></extra>'
            )
        ),
        row=2,
        col=1
    )

    # --------------------------------------------------------
    # GRÁFICO B. LOCALIDADES PRINCIPALES
    # --------------------------------------------------------

    dashboard.add_trace(
        go.Bar(

            x=r['ranking'].values,
            y=r['ranking'].index,

            orientation='h',

            marker_color=MAGENTA,

            text=r['ranking'].values,
            texttemplate='%{text:,.0f}',
            textposition='outside',

            cliponaxis=False,

            visible=visible,

            hovertemplate=(
                '<b>%{y}</b><br>'
                'Registros: %{x:,.0f}'
                '<extra></extra>'
            )
        ),
        row=2,
        col=3
    )

    # Guardar las posiciones de las cinco trazas
    indices_estados[(fuente, localidad)] = list(
        range(inicio, len(dashboard.data))
    )


# ============================================================
# 11. CREAR EL MENÚ DE FILTROS COORDINADOS
# ============================================================

# El menú actualiza simultáneamente:
#
# - Total de registros
# - Localidad principal
# - Mes principal
# - Evolución mensual
# - Ranking de localidades
# - Nota sobre cobertura temporal

botones = []

for fuente, localidad in combinaciones:

    indices = indices_estados[
        (fuente, localidad)
    ]

    # Mostrar únicamente las cinco trazas
    # correspondientes a la combinación elegida
    visibilidad = [
        i in indices
        for i in range(len(dashboard.data))
    ]

    etiqueta = (
        f'{fuente} | {localidad}'
    )

    nota_actual = resumen_dash[
        (fuente, localidad)
    ]['nota']

    botones.append(
        dict(
            label=etiqueta,

            method='update',

            args=[
                {
                    'visible': visibilidad
                },
                {
                    'title.text':
                    '05 · Análisis de registros – Línea 123',

                    # Actualizar la nota del dashboard
                    'annotations[5].text': nota_actual
                }
            ]
        )
    )


# ============================================================
# 12. APLICAR EL TEMA VISUAL
# ============================================================

tema(
    dashboard,
    '05 · Análisis de registros – Línea 123',
    altura=720
)

# ------------------------------------------------------------
# EJE DEL GRÁFICO MENSUAL
# ------------------------------------------------------------

dashboard.update_xaxes(
    title_text='Mes',
    dtick='M1',
    tickformat='%b %Y',
    showgrid=False,
    row=2,
    col=1
)

dashboard.update_yaxes(
    title_text='Total de registros',
    rangemode='tozero',
    showgrid=True,
    gridcolor=GRIS_CLARO,
    row=2,
    col=1
)

# ------------------------------------------------------------
# EJES DEL RANKING TERRITORIAL
# ------------------------------------------------------------

dashboard.update_xaxes(
    title_text='Total de registros',
    rangemode='tozero',
    showgrid=True,
    gridcolor=GRIS_CLARO,
    row=2,
    col=3
)

dashboard.update_yaxes(
    title_text=None,
    showgrid=False,
    row=2,
    col=3
)


# ============================================================
# 13. INCORPORAR LA NOTA INFERIOR
# ============================================================

# Se agrega después de crear los cinco subtítulos.
# Por eso esta anotación ocupa la posición 5.

nota_inicial = resumen_dash[
    estado_inicial
]['nota']

dashboard.add_annotation(
    x=0,
    y=-0.14,

    xref='paper',
    yref='paper',

    text=nota_inicial,

    showarrow=False,

    xanchor='left',
    align='left',

    font=dict(
        size=11,
        color=GRIS_MEDIO
    )
)


# ============================================================
# 14. AJUSTAR LA PRESENTACIÓN GENERAL
# ============================================================

dashboard.update_layout(

    showlegend=False,

    # Menú desplegable de filtros
    updatemenus=[
        dict(
            type='dropdown',
            buttons=botones,
            direction='down',

            x=0,
            y=1.13,

            xanchor='left',
            yanchor='top',

            bgcolor=BLANCO,
            bordercolor='#D9D9D9',

            font=dict(
                size=11,
                color=GRIS_OSCURO
            ),

            active=0
        )
    ],

    # Márgenes compactos y espacio para la nota
    margin=dict(
        l=105,
        r=90,
        t=150,
        b=110
    ),

    plot_bgcolor=BLANCO,
    paper_bgcolor=BLANCO,

    font=dict(
        family='Arial',
        color=GRIS_OSCURO
    ),

    hovermode='closest'
)


# ============================================================
# 15. MOSTRAR EL DASHBOARD
# ============================================================

dashboard.show()


# ============================================================
# 16. VERIFICACIÓN DE LOS RESULTADOS
# ============================================================

# Mostrar los indicadores del estado inicial
resultado_inicial = resumen_dash[
    estado_inicial
]

print("\nVERIFICACIÓN DEL DASHBOARD")

print(
    f"Total de registros: "
    f"{resultado_inicial['total']:,}"
)

print(
    f"Localidad principal: "
    f"{resultado_inicial['localidad_principal']}"
)

print(
    f"Mes de mayor volumen: "
    f"{resultado_inicial['mes_principal']}"
)

print(
    "\nSeleccione una combinación de fuente y localidad "
    "para actualizar los indicadores y gráficos."
)

## 6. Componente 5 · Exportar y compartir un HTML autocontenido
`include_plotlyjs=True` inserta JavaScript de Plotly dentro del archivo: no se necesita conexión a internet para abrirlo. Los datos agregados de las trazas quedan incluidos en el HTML; **no publicar si contienen información confidencial**.

In [ ]:
archivo='Sesion_24_Mini_Dashboard_Plotly.html'
dashboard.write_html(archivo,include_plotlyjs=True,full_html=True,auto_open=False)
print('HTML generado:',os.path.abspath(archivo),'| bytes:',os.path.getsize(archivo))
try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    print('Descargue el HTML desde la ruta indicada.')

## 7. Componente 6 · Seaborn + Plotly: un mismo flujo analítico
**Gráfico 6: comparación de propósito.** Seaborn sirve para explorar patrones con rapidez; Plotly permite entregar una vista interactiva. Ambos deben partir de **los mismos datos limpios**.

Limpieza previa: convertir fechas, eliminar solo las fechas no interpretables, agrupar por día de semana y ordenar explícitamente de lunes a domingo.

In [ ]:
# ============================================================
# GRÁFICO 6. MISMA EVIDENCIA, DOS EXPERIENCIAS DE VISUALIZACIÓN
# Comparación entre Matplotlib (estático) y Plotly (interactivo)
# Exportación de ambos gráficos a un único archivo HTML
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objects as go

import base64
import io
from IPython.display import display, HTML


# ============================================================
# 1. DEFINIR LA IDENTIDAD VISUAL
# ============================================================

# Utilizar los colores del dashboard de Power BI
MAGENTA = '#CB2F60'
GRIS_OSCURO = '#333333'
GRIS_MEDIO = '#707070'
GRIS_CLARO = '#EAEAEA'
BLANCO = '#FFFFFF'

# Establecer el orden cronológico de los días
dias_semana = [
    'Lun', 'Mar', 'Mié', 'Jue',
    'Vie', 'Sáb', 'Dom'
]


# ============================================================
# 2. LIMPIEZA ESPECÍFICA DE LOS DATOS
# ============================================================

# Seleccionar únicamente la fecha necesaria
# para construir los dos gráficos.

datos_g6 = conjunto[['fecha_analisis']].copy()

# Convertir los valores a formato datetime.
# Las fechas inválidas se convierten en NaT.

datos_g6['fecha_analisis'] = pd.to_datetime(
    datos_g6['fecha_analisis'],
    errors='coerce'
)

# Eliminar registros sin fechas válidas
datos_g6 = datos_g6.dropna(
    subset=['fecha_analisis']
).copy()

# Verificar que existan registros disponibles
assert not datos_g6.empty, (
    'No existen fechas válidas para el gráfico 6.'
)

# Identificar el día de semana de cada registro
# Python: lunes = 0 y domingo = 6

datos_g6['dia_semana'] = (
    datos_g6['fecha_analisis']
    .dt.dayofweek
)

# Contar registros por día de semana
conteos_g6 = (
    datos_g6['dia_semana']
    .value_counts()
    .reindex(range(7), fill_value=0)
)

# Crear una tabla con los datos definitivos
semanal_g6 = pd.DataFrame({
    'dia': dias_semana,
    'registros': conteos_g6.values
})

# Mostrar la tabla para verificar los resultados
print('REGISTROS POR DÍA DE SEMANA:')
print(semanal_g6.to_string(index=False))


# ============================================================
# 3. DEFINIR UNA ESCALA COMÚN PARA AMBOS GRÁFICOS
# ============================================================

# Identificar el valor máximo
maximo_g6 = semanal_g6['registros'].max()

# Agregar un margen superior del 15 %
# para que las etiquetas no queden recortadas.

limite_superior = maximo_g6 * 1.15

# Ambos gráficos utilizarán esta misma escala.


# ============================================================
# 4. GRÁFICO ESTÁTICO CON MATPLOTLIB
# ============================================================

# Crear una figura de tamaño amplio y legible
fig_estatico, ax = plt.subplots(
    figsize=(12, 5),
    dpi=120
)

# Construir las barras verticales
barras = ax.bar(
    semanal_g6['dia'],
    semanal_g6['registros'],
    color=MAGENTA,
    width=0.65
)

# Agregar etiquetas numéricas sobre las barras
for barra in barras:

    altura = barra.get_height()

    ax.annotate(
        f'{altura:,.0f}',
        xy=(
            barra.get_x() + barra.get_width() / 2,
            altura
        ),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=10,
        color=GRIS_OSCURO
    )

# Configurar título y ejes
ax.set_title(
    'Registros por día de semana',
    fontsize=17,
    color=GRIS_OSCURO,
    loc='left',
    pad=20
)

ax.set_xlabel(
    'Día de semana',
    fontsize=12,
    color=GRIS_OSCURO,
    labelpad=10
)

ax.set_ylabel(
    'Total de registros',
    fontsize=12,
    color=GRIS_OSCURO,
    labelpad=10
)

# Aplicar la misma escala que tendrá Plotly
ax.set_ylim(0, limite_superior)

# Incorporar líneas de referencia horizontales
ax.grid(
    axis='y',
    color=GRIS_CLARO,
    linewidth=0.8
)

ax.set_axisbelow(True)

# Eliminar bordes innecesarios
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.spines['bottom'].set_color(GRIS_CLARO)

# Formatear los valores del eje vertical
ax.yaxis.set_major_formatter(
    ticker.StrMethodFormatter('{x:,.0f}')
)

# Mejorar la legibilidad de las etiquetas
ax.tick_params(
    axis='both',
    labelsize=11,
    colors=GRIS_MEDIO,
    length=0
)

# Mantener fondo blanco
fig_estatico.patch.set_facecolor(BLANCO)
ax.set_facecolor(BLANCO)

# Ajustar los márgenes
fig_estatico.tight_layout(pad=2)


# ============================================================
# 5. CONVERTIR MATPLOTLIB EN UNA IMAGEN PARA EL HTML
# ============================================================

# Crear un archivo temporal en memoria
# sin necesidad de guardarlo en el computador.

buffer = io.BytesIO()

# Guardar la figura como PNG
fig_estatico.savefig(
    buffer,
    format='png',
    dpi=120,
    facecolor=BLANCO,
    bbox_inches='tight'
)

# Regresar al inicio del archivo en memoria
buffer.seek(0)

# Codificar la imagen para insertarla en HTML
imagen_base64 = base64.b64encode(
    buffer.read()
).decode('utf-8')

# Cerrar el archivo temporal
buffer.close()

# Mostrar el gráfico estático en Colab
plt.show()


# ============================================================
# 6. GRÁFICO INTERACTIVO CON PLOTLY
# ============================================================

fig_interactivo = go.Figure()

# Construir el gráfico utilizando exactamente
# los mismos datos del gráfico estático.

fig_interactivo.add_trace(
    go.Bar(

        x=semanal_g6['dia'],
        y=semanal_g6['registros'],

        marker=dict(
            color=MAGENTA
        ),

        width=0.65,

        # Mostrar valores sobre las barras
        text=semanal_g6['registros'],

        texttemplate='%{text:,.0f}',
        textposition='outside',

        textfont=dict(
            size=12,
            color=GRIS_OSCURO
        ),

        # Información interactiva al pasar el cursor
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Total de registros: %{y:,.0f}'
            '<extra></extra>'
        ),

        name='Registros'
    )
)

# ============================================================
# 7. CONFIGURAR EL DISEÑO DE PLOTLY
# ============================================================

fig_interactivo.update_layout(

    title=dict(
        text='Registros por día de semana',
        x=0.04,
        xanchor='left',
        font=dict(
            size=20,
            color=GRIS_OSCURO
        )
    ),

    # Tamaño equivalente al gráfico de Matplotlib
    width=1100,
    height=460,

    # Fondo blanco
    paper_bgcolor=BLANCO,
    plot_bgcolor=BLANCO,

    # Tipografía
    font=dict(
        family='Arial',
        size=12,
        color=GRIS_OSCURO
    ),

    # Márgenes
    margin=dict(
        l=85,
        r=45,
        t=90,
        b=75
    ),

    # Ocultar leyenda innecesaria
    showlegend=False,

    # Interacción al pasar el cursor
    hovermode='closest',

    # Permitir zoom mediante selección rectangular
    dragmode='zoom'
)

# Configurar el eje horizontal
fig_interactivo.update_xaxes(

    title_text='Día de semana',

    categoryorder='array',
    categoryarray=dias_semana,

    showgrid=False,

    showline=True,
    linecolor=GRIS_CLARO,

    tickfont=dict(size=12)
)

# Configurar el eje vertical
fig_interactivo.update_yaxes(

    title_text='Total de registros',

    range=[0, limite_superior],

    showgrid=True,
    gridcolor=GRIS_CLARO,

    zeroline=False,

    tickformat=',.0f',

    tickfont=dict(size=12)
)

# Mostrar el gráfico interactivo en Colab
fig_interactivo.show(
    config={
        'displayModeBar': True,
        'scrollZoom': True,
        'displaylogo': False,
        'responsive': True
    }
)


# ============================================================
# 8. CREAR EL DOCUMENTO HTML COMPARATIVO
# ============================================================

# Convertir Plotly en código HTML.
# El gráfico conserva sus funcionalidades interactivas.

html_plotly = fig_interactivo.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={
        'displayModeBar': True,
        'scrollZoom': True,
        'displaylogo': False,
        'responsive': True
    }
)

# Crear un documento HTML con dos secciones.
# Ambas visualizaciones tendrán el mismo ancho.

html_completo = f"""
<!DOCTYPE html>
<html lang="es">

<head>
<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1.0">

<title>
Comparación Matplotlib y Plotly
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    background-color: #F7F7F7;
    margin: 0;
    padding: 30px;
    color: #333333;
}}

.contenedor {{
    max-width: 1150px;
    margin: 0 auto;
}}

h1 {{
    font-size: 25px;
    border-left: 6px solid #CB2F60;
    padding-left: 15px;
    margin-bottom: 12px;
}}

.introduccion {{
    color: #707070;
    font-size: 15px;
    margin-bottom: 30px;
}}

.seccion {{
    background: white;
    border: 1px solid #EAEAEA;
    border-radius: 8px;
    padding: 22px;
    margin-bottom: 25px;
}}

h2 {{
    font-size: 19px;
    margin-top: 0;
    margin-bottom: 10px;
}}

.descripcion {{
    font-size: 14px;
    color: #707070;
    margin-bottom: 18px;
}}

.grafico {{
    width: 100%;
    overflow: hidden;
}}

.grafico img {{
    display: block;
    width: 100%;
    height: auto;
    margin: 0 auto;
}}

.grafico .plotly-graph-div {{
    width: 100% !important;
    max-width: 100%;
    margin: 0 auto;
}}

.conclusion {{
    background: white;
    border-left: 5px solid #CB2F60;
    padding: 20px;
    margin-top: 20px;
    font-size: 15px;
    line-height: 1.6;
}}

</style>
</head>

<body>

<div class="contenedor">

<h1>
06 · Misma evidencia, dos experiencias de visualización
</h1>

<p class="introduccion">
Comparación de un gráfico estático construido con Matplotlib
y un gráfico interactivo desarrollado con Plotly.
Ambos utilizan exactamente los mismos datos.
</p>

<div class="seccion">

<h2>1. Visualización estática — Matplotlib</h2>

<p class="descripcion">
Permite comunicar los resultados mediante una imagen fija.
No ofrece exploración interactiva de los datos.
</p>

<div class="grafico">
<img src="data:image/png;base64,{imagen_base64}"
     alt="Registros por día de semana - Matplotlib">
</div>

</div>


<div class="seccion">

<h2>2. Visualización interactiva — Plotly</h2>

<p class="descripcion">
Pase el cursor sobre las barras para consultar los valores.
Utilice la barra de herramientas para explorar el gráfico.
</p>

<div class="grafico">
{html_plotly}
</div>

</div>


<div class="conclusion">

<strong>Conclusión:</strong>

Ambas visualizaciones representan la misma información.
Matplotlib permite construir una representación estática
adecuada para informes y documentos.

Plotly incorpora herramientas de exploración,
como información emergente, zoom y desplazamiento,
que facilitan la interacción del usuario con los datos.

La elección de la herramienta debe responder al propósito
de comunicación y a las necesidades de la audiencia.

</div>

</div>

</body>
</html>
"""


# ============================================================
# 9. GUARDAR EL ARCHIVO HTML
# ============================================================

nombre_archivo = (
    'Grafico_06_Matplotlib_vs_Plotly.html'
)

# Guardar el documento en el entorno de Colab
with open(
    nombre_archivo,
    'w',
    encoding='utf-8'
) as archivo:

    archivo.write(html_completo)

print(
    f'\nArchivo HTML generado: {nombre_archivo}'
)


# ============================================================
# 10. DESCARGAR EL ARCHIVO DESDE GOOGLE COLAB
# ============================================================

# Esta instrucción permite descargar el archivo
# directamente al computador.

from google.colab import files

files.download(nombre_archivo)

## 8. Discusión y preparación del Proyecto 2
**Preguntas para el aula:**
1. ¿Qué aporta la vista integrada frente a tres gráficos separados?
2. ¿Qué principios de Gestalt justifican la proximidad de las visualizaciones y la consistencia cromática?
3. ¿Por qué no debemos comparar totales de archivos con ventanas temporales diferentes sin contextualizarlos?
4. ¿Cuándo es preferible un gráfico estático de Seaborn y cuándo una visualización interactiva de Plotly?
5. ¿Qué ocurre con el dashboard exportado si no hay internet?

**Aplicación autónoma:** cada equipo formulará una pregunta analítica para su audiencia técnica del Proyecto 2, preparará y documentará la limpieza de cada gráfico, integrará al menos dos visualizaciones interactivas, incorporará una anotación sustentada por los datos y justificará los filtros. Entregará notebook reproducible y una versión HTML si resulta pertinente.

**Lista de comprobación:**
- [ ] Variables y unidades correctas; sin duplicación por uniones.
- [ ] Limpieza reproducible inmediatamente antes de cada gráfico.
- [ ] Paleta consistente, contraste, jerarquía, títulos informativos y ejes claros.
- [ ] Filtros comprobados y limitaciones de interactividad reconocidas.
- [ ] Narrativa coherente con la audiencia y la pregunta.
- [ ] HTML probado fuera de Colab.

**Nota docente:** el menú controla vistas coordinadas, no constituye filtrado cruzado por clic entre gráficos. Esto permite enlazar pedagógicamente con Streamlit en la sesión 29.